# 💻 Notebook do Aluno — Aula 01: Revisão expressa + LangChain LCEL e ChatOllama

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 01/14 — Módulo 1: LangChain Foundations**  
**⏱️ 1h40min**  
**🐍 LCEL · ChatOllama · OutputParser**  
**🔁 Andaime 40%**  

---

## 🎯 Objetivo da aula

Construir uma chain LangChain com LCEL que substitui o chamar_llm() manual do 1º semestre — com menos código, mais composição e pronto para escalar com memória e RAG nas próximas aulas.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime do lab.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

---

## 🧩 Andaime da aula — complete as lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata
import os

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
# 👉 LACUNA 1: instancie o ChatOllama com o modelo certo
llm = ChatOllama(model=___, temperature=___)

# 👉 LACUNA 2: crie o ChatPromptTemplate com 2 roles
# system: persona + especialidade do grupo
# human: a pergunta do usuário
prompt = ChatPromptTemplate.from_messages([
    (___, ___),  # system com persona do grupo
    (___, ___),  # human com {pergunta}
])

# 👉 LACUNA 3: componha a chain com os 3 componentes
chain = ___ | ___ | ___

# 👉 LACUNA 4: invoque com uma pergunta do domínio do grupo
resposta = chain.invoke({"pergunta": ___})
print(resposta)

# Bônus: testar .stream() — observe os tokens chegando
for chunk in chain.stream({"pergunta": "Me dê um exemplo rápido."}):
    print(chunk, end="", flush=True)

---

## ✍️ Suas anotações

Registre aqui as observações da prática (qualidade dos resultados, comparações e conclusões do grupo).

---

## 🏋️ Exercícios da Aula 01

Os quatro exercícios praticam a chain LCEL completa do domínio do grupo — o mesmo domínio que persiste pelo semestre (CKP01 chatbot, CKP02 RAG, CKP03 agente). Complete os andaimes marcados com `___` e `# 👉 LACUNA`, rode cada célula no Colab e entregue o notebook executado.


### Exercício 1 — Monte a chain e inspecione os tipos · ★★☆ · 10 min

*Individual · Colab*

O pipeline LCEL tem um contrato de tipos: o template devolve mensagens formatadas, o modelo devolve `AIMessage` e o parser devolve `str`.

1. Complete o template — role de sistema com a persona do domínio e `{pergunta}` no human.
2. Componha a chain na ordem certa com o operador `|`.
3. Rode a célula e confira o tipo impresso em cada estágio — e o texto final chegando como `str`.
4. Descomente a linha do `KeyError` e rode de novo: qual variável faltou no `.invoke()`?

> **💡 Dica:** cada componente LCEL é uma `Runnable` — o `|` conecta a saída de um na entrada do próximo, como um pipeline de dados.


In [ ]:
# 👉 LACUNA: monte a chain e confirme o tipo de saída de cada estágio
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 👉 LACUNA 1: role e persona do domínio no system; {pergunta} no human
prompt = ChatPromptTemplate.from_messages([
    (___, "Você é ___ de {dominio}. Responda em 1 frase."),
    (___, "{pergunta}"),
])

# 👉 LACUNA 2: componha a chain na ordem certa (prompt → modelo → parser)
llm = ChatOllama(model="gpt-oss:120b", temperature=0.7)
chain = ___ | ___ | ___

variaveis = {"dominio": "culinária", "pergunta": "Como conservar manjericão?"}

mensagens = prompt.invoke(variaveis)
print("prompt →", type(mensagens).__name__)
print("modelo →", type(llm.invoke(mensagens.to_messages())).__name__)   # esperado: AIMessage
resposta = chain.invoke(variaveis)
print("chain  →", type(resposta))          # esperado: <class 'str'>
print(resposta)

# 👉 LACUNA 3: descomente a linha abaixo — qual variável faltou no invoke?
# chain.invoke({"dominio": "culinária"})


### Exercício 2 — Chain quebrada: análise e correção no código · ★★☆ · 10 min

*Individual · Colab*

A célula abaixo foi digitada às pressas e tem 3 defeitos: componentes na ordem errada, parser esquecido e a chave `{pergunta}` ausente no `.invoke()`.

1. Rode a célula e leia o erro — qual problema aparece primeiro?
2. Reescreva a chain corrigida nas lacunas — a ordem certa é prompt → modelo → parser.
3. Confirme com `print(type(resposta))`: o esperado é `<class 'str'>`.

> **💡 Dica:** sem `StrOutputParser()` no fim da chain, o `.invoke()` devolve um `AIMessage` — e `print()` nele mostra o objeto inteiro, não o texto.


In [ ]:
# 👉 LACUNA: a "chain quebrada" abaixo tem 3 defeitos — analise, depois corrija

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="gpt-oss:120b", temperature=0.7)

# Versão com defeitos (apenas para análise — NÃO edite este bloco):
prompt_errado = ChatPromptTemplate.from_messages([
    ("system", "Você é especialista em {tema}. Responda em 2 frases."),
    ("human",  "{pergunta}"),
])
chain_errada = llm | prompt_errado            # ❌ ordem invertida + parser ausente
resposta = chain_errada.invoke({"tema": "culinária"})   # ❌ faltou a chave {pergunta}

# 👉 LACUNA: reescreva a chain corrigida abaixo
prompt = ___
chain = ___ | ___ | ___      # ordem correta: prompt → modelo → parser
resposta = chain.invoke({"tema": "culinária", "pergunta": "Qual o segredo de um bom caldo?"})
print(type(resposta))        # esperado: <class 'str'>
print(resposta)


### Exercício 3 — Temperature e parsers: compare no código · ★★☆ · 10 min

*Individual · Colab*

Mesma pergunta, configurações diferentes — a diferença aparece no output.

1. Complete as temperatures (0 e 0.9) e rode cada chain 2 vezes: a "fria" repete a resposta? A "criativa" varia?
2. Complete a instrução de formato no system e o parser da chain JSON.
3. Confirme `type(r)` → `<class 'dict'>` e imprima as chaves com `list(r.keys())`.

> **💡 Dica:** combine `format="json"` no `ChatOllama` com a instrução no prompt — o parâmetro garante JSON sintaticamente válido, o prompt garante as chaves pedidas.


In [ ]:
# 👉 LACUNA: mesma chain, duas temperatures e dois parsers — compare no output
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

prompt_base = ChatPromptTemplate.from_messages([
    ("system", "Você é um guia de {dominio}. Responda em 1 frase."),
    ("human",  "{pergunta}"),
])

# 👉 LACUNA 1: complete as temperatures (0 e 0.9) e rode cada chain 2 vezes
chain_fria     = prompt_base | ChatOllama(model="gpt-oss:120b", temperature=___) | StrOutputParser()
chain_criativa = prompt_base | ChatOllama(model="gpt-oss:120b", temperature=___) | StrOutputParser()

for i in range(2):
    print("fria:    ", chain_fria.invoke({"dominio": "culinária", "pergunta": "Como conservar manjericão?"}))
    print("criativa:", chain_criativa.invoke({"dominio": "culinária", "pergunta": "Como conservar manjericão?"}))

# 👉 LACUNA 2: o system abaixo deve instruir "Responda SOMENTE em JSON
# com as chaves 'resposta' e 'nivel'" — complete e troque o parser
prompt_json = ChatPromptTemplate.from_messages([
    ("system", "Você é um guia de {dominio}. ___"),
    ("human",  "{pergunta}"),
])

chain_json = prompt_json | ChatOllama(model="gpt-oss:120b", format="json") | ___

r = chain_json.invoke({"dominio": "culinária", "pergunta": "Um lanche rápido"})
print(type(r))     # esperado: <class 'dict'>
print(list(r.keys()))   # esperado: ['resposta', 'nivel']
print(r)


### Exercício 4 — Chain JSON do domínio do grupo · ★★☆ · 10 min

*Individual · Colab*

Aplique LCEL + parser estruturado ao domínio escolhido pelo grupo (o mesmo do CKP01).

1. Complete a persona e a instrução de formato no system — as chaves são `resumo`, `topicos` (3 itens) e `nivel` (iniciante|intermediario|avancado).
2. Complete a chain com `ChatOllama(model="gpt-oss:120b", format="json")` e o parser estruturado.
3. Invoque com 2 perguntas reais do domínio e confirme `list(r.keys())`.

> **💡 Dica:** o `format="json"` garante JSON válido; as chaves pedidas no prompt garantem o conteúdo. Na Aula 03 o Pydantic valida o schema por inteiro.


In [ ]:
# 👉 LACUNA: chain JSON do domínio do grupo — modelo + parser estruturado
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

prompt_json = ChatPromptTemplate.from_messages([
    ("system", "Você é ___ do domínio ___. Responda SOMENTE em JSON com as chaves "
               "'resumo' (texto curto), 'topicos' (lista com 3 itens) e 'nivel' "
               "(iniciante|intermediario|avancado)."),
    ("human",  "{pergunta}"),
])

chain_json = ___ | ChatOllama(model="gpt-oss:120b", format="json") | ___

for pergunta in [___, ___]:           # 2 perguntas reais do domínio
    r = chain_json.invoke({"pergunta": pergunta})
    print(type(r))                    # esperado: <class 'dict'>
    print(list(r.keys()))             # esperado: ['resumo', 'topicos', 'nivel']
    print(r)


## 📚 Referências da aula

- Docs LangChain — LCEL (LangChain Expression Language): composição declarativa de chains com o operador |. python.langchain.com/docs/concepts/lcel
- Docs LangChain — ChatPromptTemplate: estruturar prompts com roles e variáveis. python.langchain.com/docs/concepts/prompt_templates
- Docs LangChain — Output parsers: StrOutputParser, JsonOutputParser, PydanticOutputParser. python.langchain.com/docs/concepts/output_parsers
- Docs langchain-ollama — ChatOllama: integração LangChain com modelos Ollama. python.langchain.com/docs/integrations/chat/ollama
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 2: Agentes racionais — o modelo de percepção → ação que fundamenta o conceito de pipeline de LLM.
- Ebook Polzer, D. — RAG with Python Cookbook. O'Reilly, 2025. Cap. 1: escolha de frameworks para aplicações RAG e por que LangChain (LCEL) é o padrão de mercado para orquestração.
- Ebook Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 1: Prompt Chaining — a base dos padrões agênticos; o LCEL que você aprendeu hoje é a fundação dos agentes da Aula 10.
- Ebook Lanham, M. — AI Agents in Action. Manning, 2025. Cap. 2: prompting LLMs com personas e delimitadores — reforça o que você construiu no 1º semestre.

---

**→ Próxima Aula — Aula 02 · 10/08** — Memória conversacional — Buffer, Summary e TokenBuffer
  
A chain de hoje é stateless — cada .invoke() começa do zero, igual à lista historico manual do 1º semestre, que crescia sem limite de tokens. A Aula 02 resolve isso com 3 tipos de memória — Buffer, Summary e TokenBuffer — e o CKP01 fica mais próximo.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*